<a href="https://colab.research.google.com/github/Russell-Tran/parallel_computing/blob/main/2026_09_18_cuda_walkthrough_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!nvidia-smi

Fri Sep 18 14:50:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
%%writefile hello.cu

#include <cstdio>
#include <cuda_runtime.h>

__global__ void hello()
{
  // printf("I am thread %d\n", threadIdx.x);
  printf("block %d, thread %d\n", blockIdx.x, threadIdx.x);
}

int main()
{
  // hello<<<this is the BLOCK COUNT, this is the THREAD COUNT>>>();
  // hello<<<1, 32>>>(); // 1 warp
  // hello<<<1, 128>>>(); // 4 warps
  hello<<<4, 32>>>();

  cudaDeviceSynchronize();
}

Overwriting hello.cu


In [19]:
!nvcc hello.cu -o hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!./hello

block 2, thread 0
block 2, thread 1
block 2, thread 2
block 2, thread 3
block 2, thread 4
block 2, thread 5
block 2, thread 6
block 2, thread 7
block 2, thread 8
block 2, thread 9
block 2, thread 10
block 2, thread 11
block 2, thread 12
block 2, thread 13
block 2, thread 14
block 2, thread 15
block 2, thread 16
block 2, thread 17
block 2, thread 18
block 2, thread 19
block 2, thread 20
block 2, thread 21
block 2, thread 22
block 2, thread 23
block 2, thread 24
block 2, thread 25
block 2, thread 26
block 2, thread 27
block 2, thread 28
block 2, thread 29
block 2, thread 30
block 2, thread 31
block 3, thread 0
block 3, thread 1
block 3, thread 2
block 3, thread 3
block 3, thread 4
block 3, thread 5
block 3, thread 6
block 3, thread 7
block 3, thread 8
block 3, thread 9
block 3, thread 10
block 3, thread 11
block 3, thread 12
block 3, thread 13
block 3, thread 14
block 3, thread 15
block 3, thread 16
block 3, thread 17
block 3, thread 18
block 3, thread 19
block 3, thread 20
block 3, thre

A warp is a group of 32 threads that the NVIDIA GPU schedules together.




```
Block 0
│
├── Warp 0 → threads   0–31
├── Warp 1 → threads  32–63
├── Warp 2 → threads  64–95
└── Warp 3 → threads  96–127
```



```
GRID
 └── BLOCK
      └── WARP
           └── THREAD
```



**Threads within the same block can cooperate in ways threads in different blocks generally cannot.**

Threads in one block can:

* share fast shared memory
* synchronize with __syncthreads()
* cooperate on one chunk of a larger problem

# Why we have blocks

`threadIdx.x` means:

Who am I within my block?

while `blockIdx.x` means:

Which block am I in?

## Now suppose we have 1,000,000 things to process

Imagine adding two arrays with one million elements. Our first instinct might be:

```
add<<<1, 1000000>>>();
```

But CUDA doesn't allow arbitrarily huge blocks. A block has a hardware-defined maximum number of threads—commonly 1024 threads/block on NVIDIA GPUs, including your T4.

More fundamentally, though, a block is supposed to be a local team of cooperating threads.

So instead we might choose 256 threads/block, and create enough blocks to cover our million elements (3907 blocks):

```
add<<<3907, 256>>>();
```

In other words:

```
1,000,000 pieces of work

GRID
│
├── Block 0     → 256 threads
├── Block 1     → 256 threads
├── Block 2     → 256 threads
├── ...
└── Block 3906  → 256 threads
```

And this is where GPUs become GPUs.

Your T4 doesn't have to execute block 0, then block 1, then block 2 sequentially.

It has many **SMs (Streaming Multiprocessors)**—essentially the major parallel execution units of the GPU. CUDA's scheduler can distribute blocks among them:

```
                 GPU

        ┌──────┐ ┌──────┐ ┌──────┐
        │ SM 0 │ │ SM 1 │ │ SM 2 │ ...
        └───┬──┘ └───┬──┘ └───┬──┘
            │        │         │
          Block 0  Block 1   Block 2
          Block 17 Block 23  Block 31
             ...      ...       ...
            
```
As blocks finish, more blocks can be scheduled.

That's why CUDA wants lots of independent blocks: the runtime gets freedom to distribute work across the GPU.

# Performance engineering
Suppose we need to process 1,000,000 elements. Should we use:
```
<<<31250, 32>>>
```
or
```
<<<7813, 128>>>
```
or
```
<<<3907, 256>>>
```
or
```
<<<977, 1024>>>
```
They all perform approximately the same amount of work.

But they are not necessarily equally fast.

**Up next: how the choice of block size interacts with warps, SM resources, occupancy, and the GPU's ability to hide latency.**